In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import xarray as xr
import datetime as dt
import earthaccess
# import pydap-specific tools
from pydap.client import open_url,get_cmr_urls
from pydap.client import to_netcdf as dap_to_netcdf

In [ ]:
swot_ccid = "C2799438313-POCLOUD"
time_range = [dt.datetime(2023, 2, 1), dt.datetime(2023, 2, 12)] # One month of data

# select a region in the Iceland-Faroe Ridge
bbox = [-20.760731,60.080727, -4.297294,65.675099] #[west, south, east, north]

cmr_urls = get_cmr_urls(ccid=swot_ccid, bounding_box = bbox, time_range=time_range, limit=1000) # you can incread the limit of results
print("################################################ \n We found a total of ", len(cmr_urls), "OPeNDAP URLS!!!\n################################################")

In [ ]:
auth = earthaccess.login(strategy="netrc", persist=True) # you will be promted to add your EDL credentials

# pass Token Authorization to a new Session.
my_session=auth.get_session()

### Read entire metadata

From one granule. This is important to identify variables of interested

In [ ]:
%%time
pyds = open_url(cmr_urls[0], protocol="dap4", session=my_session)
pyds.tree()

In [ ]:
Variables = [
    "/data_01/time","/data_01/longitude", "/data_01/latitude", 
     "/data_01/surface_classification_flag", "/data_01/ice_flag",
     "/data_01/mean_dynamic_topography", "/data_01/rain_flag"
]
output_path = "data/"

In [ ]:
Variables

# Download all data

Stream each remote file into its own local file, downloading only the data of interest

In [ ]:
%%time
dap_to_netcdf(cmr_urls[:20], session=my_session, keep_variables=Variables, output_path=output_path)

# Inspect a local file

In [ ]:
dt1 = xr.open_datatree(output_path+f"{cmr_urls[2].split('/')[-1]}.nc4")

In [ ]:
dt1.load()